# RetinaLens: Self-Contained Colab/Kaggle Training & Calibration Notebook

> **Objective:** Train, calibrate, and evaluate an explainable diabetic retinopathy screening model on a free GPU (Google Colab T4 / Kaggle P100).
> 
> **Hard Rules Compliance:**
> - Investigational screening aid for clinician review, NOT an automated diagnosis.
> - Reproducible random seeds (42).
> - Pinned dependencies and validated quadratic weighted kappa (QWK).
> - ONNX export with numerical verification.

## 1. Environment Setup & Dependency Installation

In [ ]:
!pip install --quiet timm>=1.0.30 albumentations>=1.4.0 grad-cam>=1.5.0 onnx>=1.16.0 onnxruntime>=1.18.0 onnxscript>=0.7.0 reportlab>=4.2.0

In [ ]:
import os
import sys
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | Device Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 2. Kaggle Dataset Setup (APTOS 2019)
Upload your `kaggle.json` API token or generate synthetic training data if running offline.

In [ ]:
# Clone or initialize RetinaLens repo
if not os.path.exists("RetinaLens"):
    !git clone https://github.com/your-username/retinalens.git RetinaLens
    %cd RetinaLens
else:
    %cd RetinaLens

sys.path.insert(0, os.getcwd())
print("Working Directory:", os.getcwd())

## 3. Train Model with Ordinal-Aware Thresholding

In [ ]:
# Run training pipeline with EfficientNet-B0 (or B3 / ResNet-50)
!python -m ml.train --config ml/configs/default_b0.yaml --epochs 15 --batch-size 16

## 4. Run Adebayo Explainability Sanity Check

In [ ]:
!python scripts/run_gradcam_sanity_check.py

## 5. Evaluate Generalization, Degradation Sweeps & Quality Gate

In [ ]:
!python -m ml.evaluate_generalization

# Display auto-generated findings
with open("results/ROBUSTNESS_FINDINGS.md", "r") as f:
    print(f.read())

## 6. Inspect Generated Charts & Artifacts

In [ ]:
from IPython.display import Image, display

display(Image("generated_charts/efficientnet_b0_calibration_diagram.png"))
display(Image("generated_charts/degradation_curves.png"))
display(Image("generated_charts/quality_gate_impact.png"))